In [ ]:
import refactoredpc as rf
import numpy as np
import plotly.graph_objects as go
from scipy.spatial import SphericalVoronoi
import matplotlib.pyplot as plt
import pandas as pd
from dataclasses import asdict
import os
from pathlib import Path

In [ ]:
# grid = np.loadtxt("./Grids/fib_grid")
# grid = np.random.rand(40000, 2)
grid = []
grid_length = 200
num_spins = 10000
jump_distance = 0.30
for i in range(grid_length):
    for j in range(grid_length):
        grid.append([i/grid_length, j/grid_length])
grid = np.array(grid)

jump_distance = np.sqrt(1/ num_spins) * 15

In [ ]:
system = rf.System(grid, jump_distance, "euclidean", num_spins=10000, delta="exact")
system.compute_neighborhood(distance_metric="l1")

Universal Neighbors

In [ ]:
# This distance is how far a cell is from the center of the univ neighbors list
def calculate_distance(x, y, radius):
    return np.sqrt((x - radius - 1) ** 2 + (y - radius - 1) ** 2)

def contribution_energy(distance, cell_jump_distance):
    return (1 + np.cos((np.pi * (distance - cell_jump_distance)) / 2)) / 4
    
def create_neighbors_list(cell_jump_distance):
    diameter = int(1 + 2 * np.ceil(cell_jump_distance) + 2)
    sphere = np.zeros([diameter, diameter])
    for i in range(diameter):
        for j in range(diameter):
            distance = calculate_distance(i, j, cell_jump_distance)
            if abs(distance - cell_jump_distance) <= 2:
                contribution = contribution_energy(distance, cell_jump_distance)
                sphere[i, j] = contribution
    return sphere

def spin_energy(neighbors_list, system, cell_jump_distance, spin):
    jump_radius = int(np.ceil(cell_jump_distance) + 1)
    system_length = len(system)
    x, y = spin[0], spin[1]
    spins_above = system_length - x - 1
    spins_below = x
    spins_right = system_length - y - 1
    spins_left = y
    if spins_above > jump_radius:
        spins_above = jump_radius
    if spins_below > jump_radius:
        spins_below = jump_radius
    if spins_left > jump_radius:
        spins_left = jump_radius
    if spins_right > jump_radius:
        spins_right = jump_radius
    return np.sum(np.multiply(system[int(x - spins_below): int(x + spins_above + 1), int(y - spins_left): int(y + spins_right + 1)], 
    neighbors_list[int(jump_radius - spins_below): int(jump_radius + spins_above + 1), int(jump_radius - spins_left): int(jump_radius + spins_right + 1)]))
    

def system_energy(neighbors_list, system, cell_jump_distance, spins):
    energy = 0
    for spin in spins:
        energy += spin_energy(neighbors_list, system, cell_jump_distance, spin)
    return energy / 2


grid_size = grid_length**2
cell_length = (1 / num_spins) ** (1 / 2)
cell_jump = jump_distance / cell_length
regular_grid = np.zeros([grid_length, grid_length])
grid_shape = (grid_length, grid_length)
neighbors = create_neighbors_list(cell_jump)

In [ ]:
system.random_init()

In [ ]:
system.thermalize(steps_per_temp=100000,verbose=True)

In [ ]:
system.greedy(reinit=False)

In [ ]:
start_temp = system.anneal_data.temperature
system.parameters.final_temp = start_temp / 200

In [ ]:
system.parameters.min_acceptance = 5e-7

In [ ]:
result = system.parallel_temper(num_replicas=8)

In [ ]:
system.anneal(save=False, verbose=True)

In [ ]:
lawn = system.grid[system.best_system]
non_lawn = system.grid[~system.best_system]
plt.scatter(lawn[:, 0], lawn[:, 1], color='green', s=1)
plt.scatter(non_lawn[:, 0], non_lawn[:, 1], color='red', s=.3)
plt.axis('equal')
plt.title("Square lattice N=40,000, 10,000 spins")
plt.show()

In [ ]:
up_spins = np.arange(grid_size)[system.best_system]
up_spins = np.unravel_index(up_spins, grid_shape)
up_spins = np.column_stack(up_spins)
regular_grid = np.zeros(grid_shape)
for spin in up_spins:
    regular_grid[spin[0], spin[1]] = 1
plt.imshow(regular_grid, cmap="Greens", interpolation="none")

In [ ]:
control_energy =system_energy(neighbors, regular_grid, cell_jump, up_spins)

In [ ]:
recalc_energy = rf.calculate_non_antipodal_system_energy(system.current_system, system.neighborhood)

In [ ]:
annealed_energy = system.current_energy

In [ ]:
all_spin_energy = 0
idxs = up_spins = np.arange(grid_size)[system.best_system]
control_energies = []
test_energies = []
for idx in idxs:
    test_energies.append(rf.calculate_non_antipodal_spin_energy(system.current_system, system.neighborhood[idx]))
    control_energies.append(spin_energy(neighbors, regular_grid, cell_jump, np.unravel_index(idx, grid_shape)))
